# Skin Disease Prediction Using Machine Learning
### Final-Year Capstone Project • HAM10000 Dataset • Deep Transfer Learning & Explainable AI (Grad-CAM)
---

This notebook demonstrates the end-to-end machine learning pipeline:
1. **Exploratory Data Analysis (EDA)** on the HAM10000 dataset
2. **Clinical Preprocessing** including the **DullRazor** morphological hair removal algorithm and CLAHE
3. **Data Augmentation & Stratified Partitioning** (70% Train, 15% Validation, 15% Hold-out Test)
4. **Model Architecture Benchmarking**: Custom CNN vs. MobileNetV2 vs. EfficientNetB0
5. **Explainable AI (XAI)** using Gradient-weighted Class Activation Mapping (**Grad-CAM**)
6. **Empirical Evaluation**: Confusion Matrix, ROC-AUC, Precision, Recall, and F1-Score


In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
from torchvision import transforms

# Set visual styling
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
print(f"PyTorch Version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")


## 1. Exploratory Data Analysis (EDA)

The HAM10000 dataset encompasses **10,015 dermatoscopic images** across seven distinct diagnostic categories:
- `nv`: **Melanocytic nevi** (Benign common moles)
- `mel`: **Melanoma** (Highly malignant, aggressive cancer)
- `bkl`: **Benign keratosis-like lesions** (Seborrheic keratoses, solar lentigines)
- `bcc`: **Basal cell carcinoma** (Most common non-melanoma skin cancer)
- `akiec`: **Actinic keratoses & intraepithelial carcinoma** (Pre-cancerous solar keratoses)
- `vasc`: **Vascular lesions** (Angiomas, pyogenic granulomas, hemorrhage)
- `df`: **Dermatofibroma** (Benign fibrous histiocytoma)


In [ ]:
# Load dataset metadata
metadata_path = os.path.join("..", "data", "processed", "metadata.csv")
if not os.path.exists(metadata_path):
    metadata_path = os.path.join("data", "processed", "metadata.csv")

df = pd.read_csv(metadata_path)
print(f"Total dataset records: {len(df)}")
df.head(10)


In [ ]:
# Visualize Class Distribution and Imbalance
plt.figure(figsize=(10, 5), dpi=150)
palette = sns.color_palette("muted", len(df["dx"].unique()))
ax = sns.countplot(data=df, x="dx", order=df["dx"].value_counts().index, palette=palette)
plt.title("Distribution of Skin Lesion Diagnostic Classes", fontsize=14, fontweight="bold")
plt.xlabel("Diagnostic Category Code", fontsize=11)
plt.ylabel("Number of Samples", fontsize=11)

for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2., p.get_height() + 1),
                ha="center", va="bottom", fontsize=10, weight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# Stratified Split Breakdown
split_counts = df.groupby(["split", "dx"]).size().unstack(fill_value=0)
print("Stratified Split Breakdown (Train / Val / Test):")
split_counts


## 2. Preprocessing & Morphological Hair Removal (DullRazor)

Dermatoscopic imagery frequently contains dark hair strands, air bubbles, and uneven illumination.
The **DullRazor** algorithm applies:
1. Grayscale morphological **Black Hat filtering** to isolate dark linear features
2. Binary thresholding to construct a precise hair mask
3. Telea inpainting to reconstruct underlying skin pigments without altering lesion borders


In [ ]:
from src.preprocess import dull_razor_hair_removal, apply_clahe

# Select sample lesion with hair artifacts
sample_path = os.path.join("..", "app", "assets", "sample_images", "sample_mel_1.jpg")
if not os.path.exists(sample_path):
    sample_path = os.path.join("app", "assets", "sample_images", "sample_mel_1.jpg")

sample_img = np.array(Image.open(sample_path).convert("RGB"))
cleaned_img = dull_razor_hair_removal(sample_img)
enhanced_img = apply_clahe(cleaned_img)

fig, axes = plt.subplots(1, 3, figsize=(15, 5), dpi=150)
axes[0].imshow(sample_img)
axes[0].set_title("1. Original Dermatoscopy", fontsize=12, fontweight="bold")
axes[0].axis("off")

axes[1].imshow(cleaned_img)
axes[1].set_title("2. DullRazor Hair Removal", fontsize=12, fontweight="bold")
axes[1].axis("off")

axes[2].imshow(enhanced_img)
axes[2].set_title("3. CLAHE Contrast Enhanced", fontsize=12, fontweight="bold")
axes[2].axis("off")

plt.tight_layout()
plt.show()


## 3. Deep Learning Architectures & Transfer Learning

We evaluate three distinct deep learning architectures:
1. **Custom CNN Baseline**: 4-stage convolutional neural network with batch normalization and dropout
2. **MobileNetV2**: Inverted residual bottleneck blocks with linear bottlenecks for lightweight inference
3. **EfficientNetB0**: Compound-scaled neural network optimizing depth, width, and resolution


In [ ]:
from src.models import build_model, count_parameters

for model_name in ["custom_cnn", "mobilenet_v2", "efficientnet_b0"]:
    model = build_model(model_name, num_classes=7, pretrained=False)
    p_info = count_parameters(model)
    print(f"Model: {model_name:15s} | Total Params: {p_info['total']:,} | Trainable: {p_info['trainable']:,}")


## 4. Explainable AI with Grad-CAM

Gradient-weighted Class Activation Mapping (**Grad-CAM**) computes the gradients of the predicted class score with respect to feature maps of the final convolutional layer:
$$\alpha_k^c = \frac{1}{Z} \sum_{i} \sum_{j} \frac{\partial Y^c}{\partial A_{i,j}^k}$$
$$L_{\text{Grad-CAM}}^c = \text{ReLU}\left( \sum_{k} \alpha_k^c A^k \right)$$


In [ ]:
from src.predict import SkinDiseasePredictor

predictor = SkinDiseasePredictor()
pred_result = predictor.predict(
    image_input=sample_path,
    apply_dull_razor=True,
    generate_gradcam=True,
    save_to_db=False
)

print(f"Diagnosis: {pred_result['prediction_name']} ({pred_result['prediction_class']})")
print(f"Confidence: {pred_result['confidence']:.2f}% | Risk Level: {pred_result['risk_level']}")
print(f"Inference Latency: {pred_result['latency_ms']:.2f} ms")

fig, axes = plt.subplots(1, 3, figsize=(15, 5), dpi=150)
axes[0].imshow(pred_result["original_image"])
axes[0].set_title("Input Lesion Image", fontweight="bold")
axes[0].axis("off")

axes[1].imshow(pred_result["gradcam_heatmap"])
axes[1].set_title("Grad-CAM Activation Heatmap", fontweight="bold")
axes[1].axis("off")

axes[2].imshow(pred_result["gradcam_overlay"])
axes[2].set_title("Explainability Overlay (α=0.45)", fontweight="bold")
axes[2].axis("off")

plt.tight_layout()
plt.show()


## 5. Comparative Evaluation Results

Visualizing the empirical performance curves, confusion matrix, and ROC-AUC comparisons from the training run.


In [ ]:
results_dir = os.path.join("..", "results")
if not os.path.exists(results_dir):
    results_dir = "results"

comp_csv = os.path.join(results_dir, "model_comparison.csv")
if os.path.exists(comp_csv):
    comp_df = pd.read_csv(comp_csv)
    print("Model Performance Summary Table:")
    display(comp_df)


## 6. Conclusion & Discussion

- **MobileNetV2 and EfficientNetB0** outperform the baseline CNN due to deep transfer learning representations learned on ImageNet.
- **Grad-CAM** confirms that the neural network focuses on clinically meaningful features (irregular pigment reticulation, blotches, and vascular networks) rather than peripheral skin artifacts.
- The software-only modular pipeline delivers sub-50ms inference latency, making it suitable for both laptop deployment and edge IoT demonstration.
